<div style="background-color: #1e1e2e; padding: 20px; border-radius: 10px; border-left: 5px solid #a6e3a1;">
    <h1 style="color: #cba6f7; margin-top: 0; font-family: sans-serif;">🚀 Application Deployment: Gradio App</h1>
    <h3 style="color: #bac2de; margin-bottom: 0; font-family: sans-serif; font-weight: normal;">Lung Cancer Detection from CT Scans Dataset</h3>
    <hr style="border: 1px solid #313244; margin-top: 15px; margin-bottom: 15px;">
    <p style="color: #a6adc8; margin-bottom: 0; font-family: sans-serif;">
        This notebook launches an interactive web application using Gradio. It loads the trained ResNet-18 and ResNet-34 models and provides a user-friendly interface for uploading CT scans and predicting the presence of lung cancer in real-time.
    </p>
</div>


### 📦 1. Importing Dependencies
First, we import the necessary libraries. We will use `gradio` for the web interface, `PIL` for image handling, and `torch`/`torchvision` to load our deep learning model.


In [ ]:
import gradio as gr
from PIL import Image
import torch
import torch.nn as nn
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights, ResNet34_Weights

### ⚙️ 2. Configuration & Model Initialization
We configure our classes (Positive and Negative) and initialize the ResNet-18 and ResNet-34 model architectures. We then load our trained weights and set the models to evaluation mode (`.eval()`).


In [ ]:
device = torch.device("cpu")
num_classes = 2

classes = ["Positive", "Negative"]

resnet18 = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
resnet18.fc = nn.Linear(resnet18.fc.in_features, num_classes)
from safetensors.torch import load_model
try:
    load_model(resnet18, "Model/resnet18_ct_v1.1.safetensors")
except Exception as e:
    print(f"Warning: Could not load ResNet18. Error: {e}")
resnet18 = resnet18.to(device)
resnet18.eval()

resnet34 = models.resnet34(weights=ResNet34_Weights.IMAGENET1K_V1)
resnet34.fc = nn.Linear(resnet34.fc.in_features, num_classes)
try:
    load_model(resnet34, "Model/resnet34_ct_v1.safetensors")
except Exception as e:
    print(f"Warning: Could not load ResNet34. Error: {e}")
resnet34 = resnet34.to(device)
resnet34.eval()

### 🖼️ 3. Image Preprocessing
Before feeding images to the model, we must preprocess them. This involves resizing to `224x224`, converting to tensors, and normalizing the pixels based on ImageNet statistics.


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


### 🧠 4. Prediction Logic
The `predict` function takes an uploaded image, transforms it, and performs a forward pass through the model to retrieve confidence probabilities for each class.


In [ ]:
def predict(image, model_choice):
    if image is None:
        return "No image provided", {}
    img = Image.fromarray(image).convert("RGB")
    img_tensor = transform(img).unsqueeze(0).to(device)
    
    selected_model = resnet18 if model_choice == "ResNet-18" else resnet34
    
    with torch.no_grad():
        output = selected_model(img_tensor)
        probabilities = torch.softmax(output, dim=1)[0]
        predicted_idx = torch.argmax(probabilities).item()
        
    label = classes[predicted_idx]
    confidence = {classes[i]: float(probabilities[i]) for i in range(num_classes)}
    return label, confidence

### 🌐 5. Building the Gradio Interface
We build an intuitive user interface using `gr.Blocks`. It features a title, a brief description, an image upload module, prediction outputs, and pre-loaded examples for quick testing.


In [ ]:
example_images = [
    ["examples/1.jpg", "ResNet-34"],
    ["examples/2.jpg", "ResNet-34"],
    ["examples/3.jpg", "ResNet-34"],
    ["examples/4.jpg", "ResNet-34"],
    ["examples/5.jpg", "ResNet-34"],
    ["examples/6.jpg", "ResNet-34"],
    ["examples/7.jpg", "ResNet-34"],
    ["examples/8.jpg", "ResNet-34"],
    ["examples/9.jpg", "ResNet-34"],
    ["examples/10.jpg", "ResNet-34"],
]

with gr.Blocks(title="Lung Cancer Detection") as demo:
    gr.Markdown(
        """
        # 🫁 Lung Cancer Detection
        Upload a lung CT scan image to detect whether it shows signs of **cancer** or is **normal**.
        You can choose between **ResNet-18** and **ResNet-34** models.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(label="Upload CT Scan Image", type="numpy")
            model_selector = gr.Dropdown(
                choices=["ResNet-18", "ResNet-34"], 
                value="ResNet-34", 
                label="Select Model"
            )
            predict_btn = gr.Button("🔍 Analyze", variant="primary")

        with gr.Column(scale=1):
            label_output = gr.Label(label="Prediction")
            confidence_output = gr.Label(label="Confidence Scores", num_top_classes=num_classes)

    gr.Examples(
        examples=example_images,
        inputs=[image_input, model_selector],
        label="Example CT Scan Images"
    )

    predict_btn.click(
        fn=predict,
        inputs=[image_input, model_selector],
        outputs=[label_output, confidence_output]
    )

### 🚀 6. Launch the App
Finally, we launch the Gradio app. By running `demo.launch()`, a local server will start and you can access the interface in your browser.


In [ ]:
if __name__ == "__main__":
    demo.launch()


### 🎉 Conclusion
The ResNet-18 and ResNet-34 models were successfully integrated into a Gradio web application. This provides a simple and effective interface for classifying CT scans as either showing signs of cancer or being normal. This application can serve as a helpful tool for preliminary screening.
